In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [3]:
from datasets import load_dataset
from huggingface_hub import notebook_login

# Login using e.g. `huggingface-cli login` to access this dataset
notebook_login()
ds = load_dataset("mitulshah/transaction-categorization")

default/train/0000.parquet:   0%|          | 0.00/71.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4501043 [00:00<?, ? examples/s]

In [5]:
train_data = ds['train']
print(f"Dataset size: {len(train_data):,}")

Dataset size: 4,501,043


In [6]:
df = ds['train'].to_pandas()[['transaction_description' , 'category' , 'country' , 'currency']]

In [7]:
df.head()

,transaction_description,category,country,currency
0,Wage,Income,USA,USD
1,Arby's (Contactless),Food & Dining,AUSTRALIA,AUD
2,Occupational Therapy,Healthcare & Medical,USA,USD
3,Potbelly Store Branch,Food & Dining,UK,GBP
4,Amazon - AUSTRALIA,Shopping & Retail,AUSTRALIA,AUD


In [8]:
df.shape

(4501043, 4)

In [9]:
# make a colum transaction description in lower case
df['transaction_description'] = df['transaction_description'].str.lower()
df['transaction_description'] = df['transaction_description'].str.replace('[^a-zA-Z ]', '', regex=True)

In [10]:
df.head()

,transaction_description,category,country,currency
0,wage,Income,USA,USD
1,arbys contactless,Food & Dining,AUSTRALIA,AUD
2,occupational therapy,Healthcare & Medical,USA,USD
3,potbelly store branch,Food & Dining,UK,GBP
4,amazon australia,Shopping & Retail,AUSTRALIA,AUD


In [11]:
cols = list(df.columns)

In [12]:
# for coverting all the values to lowercase at once
for i in cols:
    df[i] = df[i].str.lower()

In [13]:
df.head()

,transaction_description,category,country,currency
0,wage,income,usa,usd
1,arbys contactless,food & dining,australia,aud
2,occupational therapy,healthcare & medical,usa,usd
3,potbelly store branch,food & dining,uk,gbp
4,amazon australia,shopping & retail,australia,aud


In [14]:
df.isnull().sum()

,0
transaction_description,0
category,0
country,0
currency,0


In [15]:
from sklearn.preprocessing import LabelEncoder

In [16]:
# for traget variable
encoder = LabelEncoder()

In [17]:

df['category'] = encoder.fit_transform(df['category'])

In [18]:
df

,transaction_description,category,country,currency
0,wage,6,usa,usd
1,arbys contactless,3,australia,aud
2,occupational therapy,5,usa,usd
3,potbelly store branch,3,uk,gbp
4,amazon australia,7,australia,aud
...,...,...,...,...
4501038,point of sale shopping,7,usa,usd
4501039,dining out thai restaurant,3,usa,usd
4501040,,7,usa,usd
4501041,dining out sushi restaurant,3,usa,usd


In [19]:
df = pd.get_dummies(df, columns=['country', 'currency'])

In [20]:
df

,transaction_description,category,country_australia,country_canada,country_india,country_uk,country_usa,currency_aud,currency_cad,currency_gbp,currency_inr,currency_usd
0,wage,6,False,False,False,False,True,False,False,False,False,True
1,arbys contactless,3,True,False,False,False,False,True,False,False,False,False
2,occupational therapy,5,False,False,False,False,True,False,False,False,False,True
3,potbelly store branch,3,False,False,False,True,False,False,False,True,False,False
4,amazon australia,7,True,False,False,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
4501038,point of sale shopping,7,False,False,False,False,True,False,False,False,False,True
4501039,dining out thai restaurant,3,False,False,False,False,True,False,False,False,False,True
4501040,,7,False,False,False,False,True,False,False,False,False,True
4501041,dining out sushi restaurant,3,False,False,False,False,True,False,False,False,False,True


In [21]:
cols = ['country_australia', 'country_canada', 'country_india', 'country_uk', 'country_usa',
        'currency_aud', 'currency_cad', 'currency_gbp', 'currency_inr', 'currency_usd']

df[cols] = df[cols].astype(int)

In [22]:
df

,transaction_description,category,country_australia,country_canada,country_india,country_uk,country_usa,currency_aud,currency_cad,currency_gbp,currency_inr,currency_usd
0,wage,6,0,0,0,0,1,0,0,0,0,1
1,arbys contactless,3,1,0,0,0,0,1,0,0,0,0
2,occupational therapy,5,0,0,0,0,1,0,0,0,0,1
3,potbelly store branch,3,0,0,0,1,0,0,0,1,0,0
4,amazon australia,7,1,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
4501038,point of sale shopping,7,0,0,0,0,1,0,0,0,0,1
4501039,dining out thai restaurant,3,0,0,0,0,1,0,0,0,0,1
4501040,,7,0,0,0,0,1,0,0,0,0,1
4501041,dining out sushi restaurant,3,0,0,0,0,1,0,0,0,0,1


In [23]:
# Select the other numerical/one-hot encoded features from df
# Ensure these columns are of numeric type (int, float, or bool for one-hot)
numeric_features_df = df.drop(columns=['transaction_description', 'category'])

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

vectorizer = TfidfVectorizer()
description_features = vectorizer.fit_transform(df['transaction_description'])

In [25]:
description_features

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 11004857 stored elements and shape (4501043, 616)>

In [26]:

X = hstack([description_features, numeric_features_df.values])
y = df['category']

In [27]:

print(f"Shape of feature matrix X: {X.shape}")
print(f"Shape of target vector y: {y.shape}")

Shape of feature matrix X: (4501043, 626)
Shape of target vector y: (4501043,)


In [28]:
y

,category
0,6
1,3
2,5
3,3
4,7
...,...
4501038,7
4501039,3
4501040,7
4501041,3


### 1. Split Data into Training and Testing Sets

We'll split our features (`X`) and target (`y`) into training and testing sets to evaluate the model's performance on unseen data. A common split is 80% for training and 20% for testing.

In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (3600834, 626)
Shape of X_test: (900209, 626)
Shape of y_train: (3600834,)
Shape of y_test: (900209,)


In [30]:
# Importing machine learning algorithms

from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb


### 2. Initialize and Train the Model


In [31]:

models = {
    'Logistic Regression': LogisticRegression(solver='saga', max_iter=500, random_state=42, n_jobs=-1),
    # add models later
}

### 3. Evaluate the Model

Now, let's evaluate our trained model using the test set. We'll look at accuracy and a classification report for more detailed metrics.

In [39]:
from sklearn.metrics import accuracy_score, classification_report as clf_report
from sklearn.model_selection import cross_val_score

results = {}
cv_results = {}
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    report = clf_report(y_test, y_pred)
    print(report)

    cv_scores = cross_val_score(model, X, y, cv=3, n_jobs=-1)
    results[name] = acc
    cv_results[name] = cv_scores.mean()
    trained_models[name] = model

print("\nFinal Results:")
for name in models.keys():
    print(f"{name}: Test Accuracy = {results[name]:.4f}, CV Accuracy = {cv_results[name]:.4f}")

Training Logistic Regression...
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     89952
           1       1.00      1.00      1.00     90235
           2       1.00      1.00      1.00     90257
           3       1.00      0.98      0.99     89353
           4       0.98      0.97      0.98     90442
           5       0.96      0.98      0.97     89425
           6       0.98      1.00      0.99     90042
           7       0.96      0.94      0.95     90251
           8       1.00      0.98      0.99     89855
           9       0.98      1.00      0.99     90397

    accuracy                           0.99    900209
   macro avg       0.99      0.99      0.99    900209
weighted avg       0.99      0.99      0.99    900209


Final Results:
Logistic Regression: Test Accuracy = 0.9853, CV Accuracy = 0.9855


In [40]:
print("\nFinal Results:")
for name in models.keys():
    print(f"{name}: Test Accuracy = {results[name]:.4f}, CV Accuracy = {cv_results[name]:.4f}")


Final Results:
Logistic Regression: Test Accuracy = 0.9853, CV Accuracy = 0.9855


In [41]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 16005789 stored elements and shape (3600834, 626)>

In [42]:
y_pred_train = model.predict(X_train)
train_acc = accuracy_score(y_train, y_pred_train)

In [43]:
train_acc

0.985711643469263

In [44]:

import pickle

best_model = trained_models['Logistic Regression']

bundle = {
    'model': best_model,
    'vectorizer': vectorizer,   # critical — needed to transform new text
    'encoder': encoder,         # needed to decode predictions back to category names
    'feature_columns': cols     # the one-hot column order
}

with open('transaction_categorizer.pkl', 'wb') as f:
    pickle.dump(bundle, f)

print("Model bundle saved to transaction_categorizer.pkl")

Model bundle saved to transaction_categorizer.pkl
